In [ ]:
# trying to do flux generation using huggingface diffusers
# abandoned in favor of stable diffusion cpp python bindings cuz there's no docs, api is trash and generation is dogshit
# the speed is good though

from transformers import T5EncoderModel
import time
import gc
import torch
import diffusers
import argparse
from PIL import Image
import os

def flush():
    gc.collect()
    torch.cuda.empty_cache()

class Flux:
    def __init__(self):
        self.t5_encoder = T5EncoderModel.from_pretrained(
            "black-forest-labs/FLUX.1-dev", subfolder="text_encoder_2", torch_dtype=torch.bfloat16
        ).to('cuda')
        self.text_encoder = diffusers.DiffusionPipeline.from_pretrained(
            "black-forest-labs/FLUX.1-dev",
            text_encoder_2=self.t5_encoder,
            transformer=None,
            vae=None,
        ).to('cuda')
        self.pipeline = diffusers.DiffusionPipeline.from_pretrained(
            "black-forest-labs/FLUX.1-dev", 
            torch_dtype=torch.bfloat16,
            text_encoder_2=None,
            text_encoder=None,
        ).to('cuda')
        self.pipeline.load_lora_weights("~/models/img/loras/", weight_name="antiblur.safetensors")
        self.pipeline.load_lora_weights("~/models/img/loras/", weight_name="boreal.safetensors")
        self.pipeline.load_lora_weights("~/models/img/loras/", weight_name="boreal-v2.safetensors")

    @torch.inference_mode()
    def inference(self, prompt, num_inference_steps=4, guidance_scale=0.0, width=1024, height=1024, seed=1000, *args, **kwargs):
        self.text_encoder.to("cuda")
        (prompt_embeds, pooled_prompt_embeds, _) = self.text_encoder.encode_prompt(prompt=prompt, prompt_2=None, max_sequence_length=256)

        flush()
        
        output = self.pipeline(
            prompt_embeds=prompt_embeds.bfloat16(),
            pooled_prompt_embeds=pooled_prompt_embeds.bfloat16(),
            width=width,
            height=height,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            generator=torch.Generator("cuda").manual_seed(seed),
            **kwargs,
        )
        image = output.images[0]
        return image


start = time.time()
model = Flux()
print(f'loading took {time.time() - start}')


start = time.time()
image = model.inference(
    prompt="mirror selfie of a young redhead woman on top of a skyscraper [<lora:boreal:0.3>|<lora:boreal-v2:0.15>|<lora:antilbur:1>]",
    num_inference_steps=20,
    guidance_scale=7,
    width=512,
    height=768,
    seed=1000,
)
end = time.time()
image.save('output.jpg')
print(f'took {end - start}')
image.show()

In [ ]:
# ComfyUI setup
%cd  ~
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ~/ComfyUI
!pip install -r requirements.txt
%cd ~/ComfyUI/custom_nodes
!git clone https://github.com/kaibioinfo/ComfyUI_AdvancedRefluxControl.git

!huggingface-cli download city96/FLUX.1-dev-gguf flux1-dev-Q8_0.gguf --local-dir ~/ComfyUI/models/unet
!huggingface-cli download black-forest-labs/FLUX.1-dev flux1-dev.safetensors --local-dir ~/ComfyUI/models/unet

!huggingface-cli download black-forest-labs/FLUX.1-dev ae.safetensors --local-dir ~/ComfyUI/models/vae
!huggingface-cli download comfyanonymous/flux_text_encoders clip_l.safetensors --local-dir ~/ComfyUI/models/clip
!huggingface-cli download comfyanonymous/flux_text_encoders t5xxl_fp16.safetensors --local-dir ~/ComfyUI/models/clip
!huggingface-cli download Comfy-Org/sigclip_vision_384 sigclip_vision_patch14_384.safetensors --local-dir ~/ComfyUI/models/clip
!huggingface-cli download second-state/FLUX.1-Redux-dev-GGUF flux1-redux-dev.safetensors --local-dir ~/ComfyUI/models/style_models

!huggingface-cli download kudzueye/boreal-flux-dev-v2 boreal-v2.safetensors --local-dir ./models/img/loras
!huggingface-cli download kudzueye/Boreal boreal-flux-dev-lora-v04_1000_steps.safetensors --local-dir ./models/img/loras
!mv ./models/img/loras/boreal-flux-dev-lora-v04_1000_steps.safetensors ./models/img/loras/boreal.safetensors
!huggingface-cli download Shakker-Labs/FLUX.1-dev-LoRA-AntiBlur FLUX-dev-lora-AntiBlur.safetensors --local-dir ./models/img/loras
!mv ./models/img/loras/FLUX-dev-lora-AntiBlur.safetensors ./models/img/loras/antiblur.safetensors
!huggingface-cli download Aitrepreneur/FLX t5-v1_1-xxl-encoder-Q8_0.gguf --local-dir ./models/img

%cd ~
!python3 ComfyUI/main.py --port 8000

In [ ]:
# img2img amine
%cd ~
import time
start = time.time()
output = flux.img_to_img(
    prompt="""
    
    anime picture of two boys standing in a room near a table with pizza

    """,
    image="img2.jpg",
    strength=0.63,
    sample_steps=20,
    width=1024,
    height=768,
    seed=1000,
    cfg_scale=5,
    style_strength=20,
    sample_method="euler",
)
delta = round(time.time() - start)
output[0].convert('RGB').save("output_img.jpg")
from IPython.display import clear_output
clear_output()
print(f'took {delta} seconds')
output[0].show()